# Compcor Comparison
- Metrics and setup taken from [the compcor library](https://github.com/IBM/comparing-corpora) and the [meme setup](https://github.com/IBM/meme) from ["Measuring the Measuring Tools" by Kour et al.](https://doi.org/10.18653/v1/2022.gem-1.35)

In [1]:
import time
import random
import torch
import os
import sklearn
import re

import pandas as pd
import polars as pl
import numpy as np

import scipy
import statsmodels.api as sm
from statsmodels.formula.api import ols

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import compcor.corpus_metrics as corpus_metrics
from compcor.utils import Corpus
from compcor.text_tokenizer_embedder import STTokenizerEmbedder
from compcor.KSC import KSC

from datetime import datetime
from pathlib import Path
import json

import torch
import gc

from itertools import combinations

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

In [2]:
# Remove transformers verbosity to clean up space.
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

# Silence HuggingFace
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Silence Python warnings.
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("pybiber").setLevel(logging.ERROR)

In [3]:
# Make files for consistent saving.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_OUTPUT = Path("./outputCompcor") / RUN_ID

DIRS = {
    "ksc_synth": BASE_OUTPUT / "ksc_synth",
    "ksc": BASE_OUTPUT / "ksc",
    "size_imbalance": BASE_OUTPUT / "size_imbalance",
    "plots": BASE_OUTPUT / "plots",
}

# Make all directories.
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

PLOT_DIRS = {
    "ksc_synth": DIRS["plots"] / "ksc_synth",
    "ksc": DIRS["plots"] / "ksc",
    "size_imbalance": DIRS["plots"] / "size_imbalance",
}

for d in PLOT_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

In [4]:
# Set plotting visualization config options.
SMALL_SIZE = 10
matplotlib.rc('font', size=SMALL_SIZE)
matplotlib.rc('axes', titlesize=SMALL_SIZE)
sns.set_theme(style="whitegrid", font_scale=2)

In [5]:
# Add file name helper. 
def make_filename(*parts, ext="csv"):
    clean = "_".join(str(p).replace("/", "-") for p in parts)
    return f"{clean}.{ext}"

# Add plot saving helper.
def save_plot(fig, path):
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

In [6]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    # os.environ["TOKENIZERS_PARALLELISM"] = "false" # done earlier
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [7]:
# ------------------ Metric Setup (experiment_config.py) ------------------
metrics = [
	corpus_metrics.chi_square_distance,
	corpus_metrics.zipf_distance,
	corpus_metrics.classifier_distance,
	corpus_metrics.IRPR_distance,
	corpus_metrics.fid_distance,
	corpus_metrics.pr_distance,
	corpus_metrics.dc_distance,
	corpus_metrics.mauve_distance,
	corpus_metrics.traditional_biber_distance,
	corpus_metrics.zero_wasserstein_distance
]

metrics_names = [((str(dist).split()[1]).split('_')[0]).upper() for dist in metrics]
ksc_measures = ['Accuracy', 'Weighted Accuracy', 'Time', 'Monotonicity', 'Separability', 'Linearity']

# Helper function for getting metric-dependent data.
def get_metric_dependant_data(metric, corpus: Corpus):
	if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
		c = STTokenizerEmbedder().tokenize_sentences(corpus)
	elif metric in (corpus_metrics.traditional_biber_distance,corpus_metrics.zero_wasserstein_distance):
		c = corpus
	else:
		c = STTokenizerEmbedder().embed_sentences(corpus)
	return c

In [8]:
# ------------------ Loading and summarizing data functions. (utils.py) ------------------

# Simple text cleaning function.
def preprocessing(texts):
	processed_texts = []
	for text in texts:
		text = str(text).strip()
		text = re.sub(r"\s+", " ", text)
		text = re.sub(r"^\s*-\s*", "", text) # Remove dashes at the beginning of texts.
		text = re.sub(r"^\s*\d+\.\s*", "", text) # Remove numbers in 1., 2., 3. format at the beginning of the text. 
		processed_texts.append(text)
	return processed_texts

# Helper function to load a labelled corpus.
def load_corpus(filename, sep=',', max_samples=np.inf):
	data = pd.read_csv(filename, sep=sep)
	data.drop(np.where(pd.isnull(data))[0], axis=0, inplace=True) # drop null data
	data = data.apply(lambda x: x.str.strip()) # strip extra whitespace
	data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True) # randomly shuffle dataset
	if not np.isinf(max_samples):
		data = data.head(max_samples) # get the number of samples required from the dataset
	sentences = data['text'].dropna().astype(str).tolist() # convert sentences to list and remove all NaN values
	return preprocessing(sentences)

def load_generated_corpus(filename, sep=',', max_samples=np.inf):
	data = pd.read_csv(filename, sep=sep)
	data.drop(np.where(pd.isnull(data))[0], axis=0, inplace=True) # drop null data
	data = data.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x)) # strip extra whitespace
	data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True) # randomly shuffle dataset
	if not np.isinf(max_samples):
		data = data.head(max_samples) # get the number of samples required from the dataset
	sentences = data['report'].dropna().astype(str).tolist() # convert sentences to list and remove all NaN values
	return preprocessing(sentences)

def load_generated_and_real_data(max_samples=np.inf):
    atis = load_corpus('./datasets/datasetsPrep/atis.csv', max_samples=max_samples)
    atis_gen = load_generated_corpus('../../localSyntheticData/dataGeneration/processedData/atis.csv', max_samples=max_samples)

    banking77 = load_corpus('./datasets/datasetsPrep/banking77.csv', max_samples=max_samples)
    banking77_gen = load_generated_corpus('../../localSyntheticData/dataGeneration/processedData/banking77.csv', max_samples=max_samples)

    clinc150 = load_corpus('./datasets/datasetsPrep/clinc150.csv', max_samples=max_samples)
    clinc150_gen = load_generated_corpus('../../localSyntheticData/dataGeneration/processedData/clinc150.csv', max_samples=max_samples)

    clinicalDialogueSummarizations = load_corpus('./datasets/datasetsPrep/clinicalDialogueSummarizations.csv', max_samples=max_samples)
    clinicalDialogueSummarizations_gen = load_generated_corpus('../../localSyntheticData/dataGeneration/processedData/clinicalDialogueSummarizations.csv', max_samples=max_samples)

    dementiaAudio = load_corpus('./datasets/datasetsPrep/dementiaAudio.csv', max_samples=max_samples)
    dementiaAudio_gen = load_generated_corpus('../../localSyntheticData/dataGeneration/processedData/dementiaAudio.csv', max_samples=max_samples)

    huffPostNews = load_corpus('./datasets/datasetsPrep/huffPostNews.csv', max_samples=max_samples)
    huffPostNews_gen = load_generated_corpus('../../localSyntheticData/dataGeneration/processedData/huffPostNews.csv', max_samples=max_samples)

    medicalAbstracts = load_corpus('./datasets/datasetsPrep/medicalAbstracts.csv', max_samples=max_samples)
    medicalAbstracts_gen = load_generated_corpus('../../localSyntheticData/dataGeneration/processedData/medicalAbstracts.csv', max_samples=max_samples)

    simSUM = load_corpus('./datasets/datasetsPrep/simSUM.csv', max_samples=max_samples)
    simSUM_gen = load_generated_corpus('../../localSyntheticData/dataGeneration/processedData/simSUM.csv', max_samples=max_samples)

    syntheticCareHomeNurseNotes = load_corpus('./datasets/datasetsPrep/syntheticCareHomeNurseNotes.csv', max_samples=max_samples)
    syntheticCareHomeNurseNotes_gen = load_generated_corpus('../../localSyntheticData/dataGeneration/processedData/syntheticCareHomeNurseNotes.csv', max_samples=max_samples)

    yahoo = load_corpus('./datasets/datasetsPrep/yahoo.csv', max_samples=max_samples)
    yahoo_gen = load_generated_corpus('../../localSyntheticData/dataGeneration/processedData/yahoo.csv', max_samples=max_samples)

    return (
        atis, atis_gen,
        banking77, banking77_gen,
        clinc150, clinc150_gen,
        clinicalDialogueSummarizations, clinicalDialogueSummarizations_gen,
        dementiaAudio, dementiaAudio_gen,
        huffPostNews, huffPostNews_gen,
        medicalAbstracts, medicalAbstracts_gen,
        simSUM, simSUM_gen,
        syntheticCareHomeNurseNotes, syntheticCareHomeNurseNotes_gen,
        yahoo, yahoo_gen
    )

# Helper function to summarize results.
def summarize_results(metrics_measures_df):
	mu = metrics_measures_df.groupby(['metric']).mean()
	mu = mu.round(decimals=3)
	std = metrics_measures_df.groupby(['metric']).std()
	return mu, std

In [9]:
# ------------------ Functions to compute metric characteristics. (metric_characteristics.py) ------------------

# Helper function for metric monotonicity.
def metric_monotonicity(ells, distances):
	return scipy.stats.spearmanr(ells, distances).correlation

# Helper function for metric separability.
def metric_separability(ells, distances):
	df = pd.DataFrame(data=list(zip(ells, distances)), columns=['ell', 'distance'])
	model = ols('distance ~ C(ell)', data=df).fit()
	aov_table = sm.stats.anova_lm(model, typ=2)
	return anova_table(aov_table).loc['C(ell)', 'omega_sq']

# Helper function for anova table.
def anova_table(aov):
	aov['mean_sq'] = aov[:]['sum_sq'] / aov[:]['df']
	aov['eta_sq'] = aov.iloc[:-1]['sum_sq'] / sum(aov['sum_sq'])
	aov['omega_sq'] = (aov.iloc[:-1]['sum_sq'] - (aov.iloc[:-1]['df'] * aov['mean_sq'].iloc[-1])) / (
                sum(aov['sum_sq']) + aov['mean_sq'].iloc[-1])
	cols = ['sum_sq', 'df', 'mean_sq', 'F', 'PR(>F)', 'eta_sq', 'omega_sq']
	aov = aov[cols]
	return aov

# Helper function for metric linearity.
def metric_linearity(ells, distances):
	return scipy.stats.linregress(ells, y=distances).rvalue

# Helper function for robustness.
def metric_size_robustness(sizes, distances, true_distance):
	# Add epsilon to prevent zero divide. 
	epsilon = 1e-8
	return 1 - np.nansum(np.abs((distances - true_distance))) / ((true_distance + epsilon) * 10 * len(np.unique(sizes)))

# Helper function for metric imbalance robustness.
def metric_imbalance_robustness(sizes, comp_sizes, distances, true_distance):
	return 1 - sum(np.abs((distances - true_distance))) / (10 * len(np.unique(sizes)))

In [10]:
def runKSC(metrics, metric_names, corpus1, corpus2,  output_dir, n=30, k=7, repetitions=5, output_name = 'test'):
	ksc_results = []
	distance_results = []

	output_dir.mkdir(parents=True, exist_ok=True)

	for metric_idx, metric in enumerate(metrics):
		with torch.no_grad():
			c1 = get_metric_dependant_data(metric, corpus1)
			c2 = get_metric_dependant_data(metric, corpus2)
		if torch.is_tensor(c1):
			c1 = c1.detach().cpu()
		if torch.is_tensor(c2):
			c2 = c2.detach().cpu()
		for rep in range(repetitions):

			distances_metric = []

			with torch.no_grad():
				ksc = KSC._known_similarity_corpora(c1, c2, n=n, k=k, unique_samples_corpora=True)
			start = time.time()
			with torch.no_grad():
				accuracy, weighted_accuracy, distance_stats = KSC.test_ksc(ksc, dist=metric)
			ksc_time = (time.time() - start) / len(distance_stats)

			distances_metric.append(
				np.vstack([[metric_names[metric_idx], rep, a, b, b - a, y] for (a, b, y) in distance_stats]))

			distances_metric = np.vstack(distances_metric)

			# normalize the score for a specific metric.
			distances_metric = np.append(distances_metric, sklearn.preprocessing.StandardScaler().fit_transform(
				distances_metric[:, 5].reshape(-1, 1)), axis=1)
			distance_results.extend(distances_metric)

			ells = distances_metric[:, 4].astype('float')
			ds_normalized = distances_metric[:, 6].astype('float')

			monotonicity = metric_monotonicity(ells, ds_normalized)
			separability = metric_separability(ells, ds_normalized)
			linearity = metric_linearity(ells, ds_normalized)
			ksc_results.append(
                [metric_names[metric_idx], accuracy, weighted_accuracy, ksc_time, monotonicity, separability, linearity])
			
			gc.collect()
			torch.cuda.empty_cache()
			
		del c1, c2, ksc, distance_stats

	metrics_measures_df = pd.DataFrame(data=ksc_results, columns=['metric'] + ksc_measures)
	metrics_measures_df['Time'] = (1 / metrics_measures_df['Time'])/100

	all_distance_samples_df = pd.DataFrame(data=distance_results,
                                	columns=['metric', 'repetition', 'i', 'j', 'l', 'distance', 'distance_score'])
	all_distance_samples_df["l"] = pd.to_numeric(all_distance_samples_df["l"])
	all_distance_samples_df["distance"] = pd.to_numeric(all_distance_samples_df["distance"])
	all_distance_samples_df["distance_score"] = pd.to_numeric(all_distance_samples_df["distance_score"])
	metrics_measures_df.to_csv(
    output_dir / make_filename(f"{output_name}_ksc_metrics_measures"), index=False
	)

	all_distance_samples_df.to_csv(
		output_dir / make_filename(f"{output_name}_ksc_distance_samples"), index=False
	)

	return metrics_measures_df, all_distance_samples_df


def plotKSC(all_distance_samples_df, save_path = None, output_name='test'):
	metrics_names = np.unique(all_distance_samples_df['metric'])
	fig, axlist = plt.subplots(1, len(metrics_names), figsize=(35, 5))
	if len(metrics_names) == 1:
		axlist = [axlist]
	for i, metric in enumerate(metrics_names):
		metric_df = all_distance_samples_df[all_distance_samples_df['metric'] == metric]
		sns.scatterplot(x='l', y='distance', data=metric_df, ax=axlist[i], color='orange')
		sns.regplot(x='l', y='distance', data=metric_df, ax=axlist[i],
					scatter=False, truncate=False)
		axlist[i].set_title('{}'.format(metric))
		axlist[i].set_xlabel('')
		axlist[i].set_ylabel('')

	plt.subplots_adjust(left=0.05,
                        bottom=0.1,
                        right=0.99,
                        top=0.9,
                        wspace=0.3,
                        hspace=0.4)

	if save_path:
		save_plot(fig, save_path / f"{output_name}_ksc_distance_plot.png")
	else:
		plt.show()


def plot_measures_results(metrics_measures_df, save_path = None, output_name='test'):
	fig, ax = plt.subplots(1, 6, figsize=(35, 5))
	if isinstance(ax, np.ndarray):
		ax = ax.flatten()
	else:
		ax = [ax]
	for i, measure in enumerate(ksc_measures):
		sns.boxplot(ax=ax[i], x='metric', y=measure, data=metrics_measures_df)
		ax[i].set_xlabel('')
		ax[i].tick_params(axis='x', labelsize=5)

	plt.subplots_adjust(left=0.1,
                        bottom=0.1,
                        right=0.99,
                        top=0.9,
                        wspace=0.3,
                        hspace=0.4)

	if save_path:
		save_plot(fig, save_path / f"{output_name}_ksc_measures_boxplot.png")
	else:
		plt.show()


# n_samples = 100
n_samples = 40 # as the smallest real dataset tested is 549 data points long, so 40 x 12 will be 480
L = [n_samples, 7]
H = [n_samples, 12]
rep = 5

max_samples = H[0] * H[1] * rep

# Make data for KSC experiment.
(atis, atis_gen, 
banking77, banking77_gen, 
clinc150, clinc150_gen, 
clinicalDialogueSummarizations, clinicalDialogueSummarizations_gen,
dementiaAudio, dementiaAudio_gen,
huffPostNews, huffPostNews_gen,
medicalAbstracts, medicalAbstracts_gen,
simSUM, simSUM_gen,
syntheticCareHomeNurseNotes, syntheticCareHomeNurseNotes_gen,
yahoo, yahoo_gen
) = load_generated_and_real_data(max_samples)

# Make dictionaries. 
real_datasets = [
    ('atis', atis),
    ('banking77', banking77),
    ('clinc150', clinc150),
    ('clinicalDialogueSummarizations', clinicalDialogueSummarizations),
    ('dementiaAudio', dementiaAudio),
    ('huffPostNews', huffPostNews),
    ('medicalAbstracts', medicalAbstracts),
    ('simSUM', simSUM),
    ('syntheticCareHomeNurseNotes', syntheticCareHomeNurseNotes),
    ('yahoo', yahoo),
]

gen_datasets = [
    ('atis_gen', atis_gen),
    ('banking77_gen', banking77_gen),
    ('clinc150_gen', clinc150_gen),
    ('clinicalDialogueSummarizations_gen', clinicalDialogueSummarizations_gen),
    ('dementiaAudio_gen', dementiaAudio_gen),
    ('huffPostNews_gen', huffPostNews_gen),
    ('medicalAbstracts_gen', medicalAbstracts_gen),
    ('simSUM_gen', simSUM_gen),
    ('syntheticCareHomeNurseNotes_gen', syntheticCareHomeNurseNotes_gen),
    ('yahoo_gen', yahoo_gen),
]


real_pairs = list(combinations(real_datasets, 2))
real_gen_pairs = list(zip(real_datasets, gen_datasets))

for R in [L,H]:
	for (name1, d1), (name2, d2) in real_pairs:
		results_file_name = DIRS["ksc"] / f"{name1}_{name2}_{R[0]}_{R[1]}"
		output_name = f"{name1}_{name2}_{R[0]}_{R[1]}"
		metrics_measures_df, all_distance_samples_df = runKSC(metrics, metrics_names, d1, d2,  output_dir=results_file_name, n=R[0], k=R[1], repetitions=rep, output_name = output_name)

		plotKSC(all_distance_samples_df, save_path=PLOT_DIRS["ksc"], output_name = output_name)
		plot_measures_results(metrics_measures_df, save_path=PLOT_DIRS["ksc"], output_name = output_name)
		mu12, std12 = summarize_results(metrics_measures_df)

		del metrics_measures_df, all_distance_samples_df
		torch.cuda.ipc_collect()
		torch.cuda.empty_cache()
		gc.collect()
		
# Make data for KSC experiment with synthetic data.
for R in [L,H]:
	for (name1, d1), (name2, d2) in real_gen_pairs:
		results_file_name = DIRS["ksc_synth"] / f"{name1}_{name2}_{R[0]}_{R[1]}"
		output_name = f"{name1}_{name2}_{R[0]}_{R[1]}"
		metrics_measures_df, all_distance_samples_df = runKSC(metrics, metrics_names, d1, d2,  output_dir=results_file_name, n=R[0], k=R[1], repetitions=rep, output_name = output_name)

		plotKSC(all_distance_samples_df, save_path=PLOT_DIRS["ksc_synth"], output_name = output_name)
		plot_measures_results(metrics_measures_df, save_path=PLOT_DIRS["ksc_synth"], output_name = output_name)
		mu12, std12 = summarize_results(metrics_measures_df)

		del metrics_measures_df, all_distance_samples_df
		torch.cuda.ipc_collect()
		torch.cuda.empty_cache()
		gc.collect()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Num judgments: 156
chi: KSC_Score: 0.0,Weighted KSC:0.0
Num judgments: 156
chi: KSC_Score: 0.0,Weighted KSC:0.0
Num judgments: 156
chi: KSC_Score: 0.01282051282051282,Weighted KSC:0.020246330352623586
Num judgments: 156
chi: KSC_Score: 0.0,Weighted KSC:0.0
Num judgments: 156
chi: KSC_Score: 0.02564102564102564,Weighted KSC:0.04049266070524717


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_26_past_participle', 'f_50_discourse_particles', 'f_58_verb_seem']
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_22_that_adj_comp', 'f_26_past_participle', 'f_33_pied_piping', 'f_50_discourse_particles']
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_22_that_adj_comp', 'f_26_past_participle', 'f_33_pied_piping', 'f_50_discourse_particles']
[INFO] Using TTR for f_43_type_token
[INFO] All features n

Num judgments: 156
traditional: KSC_Score: 0.9551282051282052,Weighted KSC:0.9291378437658174


[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_12_proverb_do', 'f_26_past_participle', 'f_32_wh_obj', 'f_50_discourse_particles', 'f_58_verb_seem']
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_12_proverb_do', 'f_26_past_participle', 'f_32_wh_obj', 'f_50_discourse_particles']
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_32_wh_obj', 'f_50_discourse_particles', 'f_58_verb_seem']
[INFO] Using TTR for f_43_type_token
[INFO] All features normaliz

Num judgments: 156
traditional: KSC_Score: 0.9615384615384616,Weighted KSC:0.9392610089421293
Num judgments: 156
zero: KSC_Score: 0.7948717948717948,Weighted KSC:0.7216129576514256
Num judgments: 156
zero: KSC_Score: 0.8974358974358975,Weighted KSC:0.8380293571790113
Num judgments: 156
zero: KSC_Score: 0.9102564102564102,Weighted KSC:0.8683988527079467
Num judgments: 156
zero: KSC_Score: 0.8076923076923077,Weighted KSC:0.7334233170237895


KeyboardInterrupt: 

In [ ]:
# ------------------ Functions to compute size imbalance experiments. (size_imbalance_experiment.py) ------------------
def size_imbalance_sensitivity_experiment(metrics, metrics_names, corpus1, corpus2, sizes, repetitions, output_folder, corpus1_name, corpus2_name):
	distance_results = []
	source_corpora_distance = []
	for metric_idx, metric in enumerate(metrics):
		metric_distances = []
		for rep in range(repetitions):
			c1 = get_metric_dependant_data(metric, corpus1)
			c2 = get_metric_dependant_data(metric, corpus2)

			for (s, sc) in zip(sizes, reversed(sizes)):
				indices = random.sample(range(len(c1)), s)
				set1 = [c1[i] for i in indices]

				indices = random.sample(range(len(c2)), sc)
				set2 = [c2[i] for i in indices]

				indices = random.sample(range(len(c2)), s)
				set2_same_size = [c2[i] for i in indices]

				dist_complemeting = metric(set1, set2)
				dist_same_size = metric(set1, set2_same_size)

				metric_distances.append([metrics_names[metric_idx], rep, s, sc, dist_same_size, dist_complemeting])

		metric_distances_df = pd.DataFrame(metric_distances, columns=[
			'metric', 'repetition', 'size', 'size_complementing',
			'distance(same)', 'distance(comp)'
		])

		scaler_same = sklearn.preprocessing.StandardScaler().fit(metric_distances_df[['distance(same)']].values)
		scaler_comp = sklearn.preprocessing.StandardScaler().fit(metric_distances_df[['distance(comp)']].values)

		metric_distances_df['distance(same)_norm'] = scaler_same.transform(metric_distances_df[['distance(same)']].values)
		metric_distances_df['distance(comp)_norm'] = scaler_comp.transform(metric_distances_df[['distance(comp)']].values)

		distance_results.append(metric_distances_df)

		sources_distance = metric(c1, c2)
		sources_distance_norm = scaler_same.transform(np.array([[sources_distance]]))[0][0]

		source_corpora_distance.append([
			metrics_names[metric_idx], sources_distance, sources_distance_norm
		])

	size_imbalance_df = pd.concat(distance_results, ignore_index=True)
	source_corpora_distance_df = pd.DataFrame(source_corpora_distance, columns=[
		'metric', 'distance', 'distance_norm'
	])

	size_imbalance_df.to_csv(
		output_folder / make_filename(corpus1_name, corpus2_name, "size_imbalance"),
		index=False
	)

	source_corpora_distance_df.to_csv(
		output_folder / make_filename(corpus1_name, corpus2_name, "source_distance"),
		index=False
	)
	return size_imbalance_df, source_corpora_distance_df


def size_imbalance_robustness_measure(size_imbalance_df, source_corpora_distance_df):
	source_corpora_distance_df['size_robustness'] = np.empty(len(source_corpora_distance_df))
	source_corpora_distance_df['imbalance_robustness'] = np.empty(len(source_corpora_distance_df))
	for metric_name in np.unique(size_imbalance_df['metric']):
		metric_sizes_distance_samples = size_imbalance_df[size_imbalance_df['metric'] == metric_name]
		metric_true_sources_distance = \
		source_corpora_distance_df[source_corpora_distance_df['metric'] == metric_name]['distance'].iloc[0]

		metric_size_sens = metric_size_robustness(list(metric_sizes_distance_samples['size']),
													list(metric_sizes_distance_samples['distance(same)']),
													metric_true_sources_distance)

		metric_imbalance_sens = metric_imbalance_robustness(list(metric_sizes_distance_samples['size']),
															list(metric_sizes_distance_samples['size_complementing']),
															metric_sizes_distance_samples['distance(comp)'],
															metric_true_sources_distance)

		source_corpora_distance_df.loc[
			source_corpora_distance_df['metric'] == metric_name, 'size_robustness'] = metric_size_sens
		source_corpora_distance_df.loc[
			source_corpora_distance_df['metric'] == metric_name, 'imbalance_robustness'] = metric_imbalance_sens

	return size_imbalance_df, source_corpora_distance_df


# columns = 'distance(comp)_norm' or 'distance(same)_norm'
def plot_size_imbalance_scatter(df_distances, df_distance_corpora_all, column='distance(same)_norm', save_path = None, output_name = 'test'):
	# Save a palette to a variable:
	palette = sns.color_palette("Paired")
	metrics_names = np.unique(df_distances['metric'])
	x_min_max = [np.min(df_distances['size']), np.max(df_distances['size'])]
	fig, ax = plt.subplots(1, len(metrics_names), figsize=(len(metrics_names) * 5, 5))
	if len(metrics_names) == 1:
		ax = [ax]
	for i, metric in enumerate(metrics_names):
		sns.scatterplot(x='size', y=column, data=df_distances[df_distances['metric'] == metric],
						ax=ax[i], color=palette[1], s=50)
		df_metric = df_distance_corpora_all[df_distance_corpora_all['metric'] == metric]
		mean_distance = np.mean(df_metric['distance'])
		ax[i].axhline(y=mean_distance, color=palette[2], linewidth=3)
		ax[i].set_title(f'{metric}', fontsize=14)
		ax[i].set_xlabel(None)
		ax[i].set_ylabel(None)
		ax[i].tick_params(axis='x', labelsize=5)
		ax[i].tick_params(axis='y', labelsize=5)

	plt.subplots_adjust(left=0.05,
                        bottom=0.1,
                        right=0.99,
                        top=0.9,
                        wspace=0.3,
                        hspace=0.4)

	# plt.savefig('size_sens.png')
	if save_path:
		save_plot(fig, save_path / f"{output_name}_size_imbalance_{column}.png")
	else:
		plt.show()


N = 500
repetitions = 10
start = 10
step = 35

for (name1, d1), (name2, d2) in real_pairs:
    size_imbalance_df, source_corpora_distance_df = size_imbalance_sensitivity_experiment(metrics, metrics_names, d1, d2,
                                                                                            list(range(start, N + 1, step)), repetitions, DIRS["size_imbalance"], name1, name2)

    source_corpora_distance_df = source_corpora_distance_df.sort_values(by="metric", ascending=1)

    size_imbalance_df, source_corpora_distance_df = size_imbalance_robustness_measure(size_imbalance_df, source_corpora_distance_df)
    plot_size_imbalance_scatter(size_imbalance_df, source_corpora_distance_df, column='distance(same)', save_path=PLOT_DIRS["size_imbalance"], output_name=f"{name1}_{name2}")
    plot_size_imbalance_scatter(size_imbalance_df, source_corpora_distance_df,column='distance(comp)', save_path=PLOT_DIRS["size_imbalance"], output_name=f"{name1}_{name2}")

    del size_imbalance_df, source_corpora_distance_df
    torch.cuda.ipc_collect()
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
# Save config.
CONFIG = {
    "run_id": RUN_ID,
    "random_state": RANDOM_STATE,

    "ksc_synth": {
        "description": "banking/huff synthetic comparison"
    },

    "ksc": {
        "L": L,
        "H": H,
        "repetitions": rep,
    },

    "size_imbalance": {
        "N": N,
        "start": start,
        "step": step,
        "repetitions": repetitions,
    }
}
(BASE_OUTPUT / "config.json").write_text(json.dumps(CONFIG, indent=2))